# Andre Karpathy GPU Speedup — 2K render (Remotion)

This notebook renders the **2K QHD master (2560×1440, 30 fps, with audio)** and writes everything to **Google Drive**.

- The first cell mounts Drive and asks for access.
- The render streams **real-time progress** (percent, rendered frames, elapsed, ETA) into the cell output and a Drive log.
- The 2K render is CPU-bound. A standard Colab CPU runtime works but takes a while; High-RAM helps. A GPU runtime does not speed up browser rendering.
- Nothing is written to Drive until you run the Drive cell and approve access.

In [ ]:
#@title 1) Mount Google Drive (permission prompt) and choose the output folder { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')

#@markdown Change this if you want a different Drive folder.
DRIVE_ROOT = "/content/drive/MyDrive/andrej_karpathy_speedup"  #@param {type:"string"}

from pathlib import Path
DRIVE_ROOT = Path(DRIVE_ROOT).expanduser()
RENDERS_DIR = DRIVE_ROOT / "renders"
LOGS_DIR = DRIVE_ROOT / "logs"
RENDERS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)
print("Drive access granted.")
print("Renders:", RENDERS_DIR)
print("Logs:   ", LOGS_DIR)

In [ ]:
#@title 2) Clone or update the GitHub project
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/lroe/andrej_karpathy_speedup.git"
PROJECT_DIR = Path("/content/andrej_karpathy_speedup")

def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Command failed: {cmd}")

if PROJECT_DIR.exists():
    run(["git", "-C", str(PROJECT_DIR), "fetch", "--all", "--prune"])
    run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"])
else:
    run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)])

run(["git", "-C", str(PROJECT_DIR), "log", "-1", "--oneline"])
print("Project:", PROJECT_DIR)

In [ ]:
#@title 3) Install Node, npm packages, and the Remotion browser
import os, shutil, subprocess, sys

def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Command failed: {cmd}")

if shutil.which("node") is None:
    run(["bash", "-lc", "curl -fsSL https://deb.nodesource.com/setup_22.x | bash -"])
    run(["apt-get", "install", "-y", "nodejs"])

# Chromium/Chrome libraries used by Remotion headless shell.
# These are harmless if already present.
run(["bash", "-lc", "apt-get update -qq && apt-get install -y -qq libnss3 libatk-bridge2.0-0 libgtk-3-0 libgbm1 libasound2 libxshmfence1 libxdamage1 libxrandr2 libxkbcommon0 libpangocairo-1.0-0 || true"])

run(["node", "--version"])
run(["npm", "--version"])
run(["npm", "ci"], cwd=PROJECT_DIR)
run(["npx", "remotion", "browser", "ensure"], cwd=PROJECT_DIR)
print("Runtime ready.")

In [ ]:
#@title 4) Render the 2K master with live progress { display-mode: "form" }
import os, subprocess, time
from pathlib import Path
from datetime import datetime

OVERWRITE = True  #@param {type:"boolean"}
PROGRESS_LOG = LOGS_DIR / f"render-2k-{datetime.now().strftime('%Y%m%d-%H%M%S')}.log"
OUTPUT_MP4 = RENDERS_DIR / "andrej_karpathy_speedup_2k.mp4"

print("Output:", OUTPUT_MP4)
print("Log:   ", PROGRESS_LOG)
print("=" * 70, flush=True)

if OUTPUT_MP4.exists() and not OVERWRITE:
    print("File already exists. Set OVERWRITE=True to render again.")
else:
    if OUTPUT_MP4.exists():
        OUTPUT_MP4.unlink()
    env = os.environ.copy()
    env["REMOTION_CONCURRENCY"] = str(max(1, min(4, os.cpu_count() or 2)))
    env["REMOTION_CRF"] = "18"
    cmd = ["node", "scripts/render-2k.mjs", str(OUTPUT_MP4), "GpuKitchen2K"]
    started = time.time()
    p = subprocess.Popen(
        cmd, cwd=str(PROJECT_DIR), env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    with open(PROGRESS_LOG, "w", encoding="utf-8") as log:
        for line in p.stdout:
            print(line, end="", flush=True)
            log.write(line); log.flush()
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"Render failed with exit code {rc}. See {PROGRESS_LOG}")
    print("=" * 70)
    print(f"Render complete in {(time.time()-started)/60:.1f} min")
    print(f"MP4: {OUTPUT_MP4}")
    print(f"Log: {PROGRESS_LOG}")

In [ ]:
#@title 5) Verify the file and archive the project + notebook to Drive
import json, os, shutil, subprocess, zipfile
from pathlib import Path

print("MP4 exists:", OUTPUT_MP4.exists())
if OUTPUT_MP4.exists():
    print("Size: %.1f MB" % (OUTPUT_MP4.stat().st_size / (1024*1024)))
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "stream=width,height,codec_name",
         "-show_entries", "format=duration,size", "-of", "json", str(OUTPUT_MP4)],
        capture_output=True, text=True
    )
    print(probe.stdout)

# Copy the notebook and a source archive (without node_modules/out/.git) to Drive.
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
notebook_candidates = list(PROJECT_DIR.glob("notebooks/*.ipynb"))
for nb in notebook_candidates:
    shutil.copy2(nb, DRIVE_ROOT / nb.name)
    print("Notebook saved:", DRIVE_ROOT / nb.name)

archive = DRIVE_ROOT / "andrej_karpathy_speedup_source.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for path in PROJECT_DIR.rglob("*"):
        if not path.is_file():
            continue
        rel = path.relative_to(PROJECT_DIR)
        if any(part in {"node_modules", "out", ".git", ".remotion"} for part in rel.parts):
            continue
        if path.stat().st_size > 80 * 1024 * 1024:
            continue
        z.write(path, rel)
print("Source archive:", archive, "%.1f MB" % (archive.stat().st_size / (1024*1024)))

## Notes

- The master is written to `MyDrive/andrej_karpathy_speedup/renders/andrej_karpathy_speedup_2k.mp4`.
- The progress log is written beside it under `logs/`.
- To change quality, edit `REMOTION_CRF` in the render cell (18 is visually clean; lower = larger file).
- To change resolution, edit `GpuKitchen2K` in `src/Root.tsx` / the render call.
- If Colab disconnects, rerun the setup cells and resume only the render cell. Rendering is not resumable frame-by-frame, but Drive outputs are preserved.